# ADA 12h/24h HMM overview

Обзор артефактов `train_ada_hmm_12h_24h_model.py`: метрики обучения, occupancy, длительности режимов, transition matrix, профили состояний и распределение состояний по времени. Все данные читаются из S3.

In [ ]:
import io
import json
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from build_price_feature_day import make_s3_client

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
BUCKET = 'binance-data-downloader'
OUTPUT_PREFIX = 'features/hmm_models/ada_hmm_12h_24h'
RUN_ID = None  # например: 'ada_hmm_12h_24h_20260714_120000'; None = последний run_id

s3 = make_s3_client()

def list_keys(prefix: str) -> list[str]:
    keys = []
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix.rstrip('/') + '/'):
        keys.extend(item['Key'] for item in page.get('Contents', []))
    return keys

def latest_run_id() -> str:
    keys = list_keys(f'{OUTPUT_PREFIX}/runs')
    run_ids = sorted({key.split('/run_id=', 1)[1].split('/', 1)[0] for key in keys if '/run_id=' in key})
    if not run_ids:
        raise FileNotFoundError(f'No runs found under s3://{BUCKET}/{OUTPUT_PREFIX}/runs/')
    return run_ids[-1]

run_id = RUN_ID or latest_run_id()
RUN_PREFIX = f'{OUTPUT_PREFIX}/runs/run_id={run_id}'
print('run_id:', run_id)
print('s3 prefix:', f's3://{BUCKET}/{RUN_PREFIX}/')


In [ ]:
def read_bytes(key: str) -> bytes:
    return s3.get_object(Bucket=BUCKET, Key=key)['Body'].read()

def read_parquet(key: str) -> pd.DataFrame:
    return pd.read_parquet(io.BytesIO(read_bytes(key)))

result = read_parquet(f'{RUN_PREFIX}/result.parquet')
profiles = read_parquet(f'{RUN_PREFIX}/state_profiles.parquet')
metadata = json.loads(read_bytes(f'{RUN_PREFIX}/metadata.json').decode('utf-8'))

try:
    states = read_parquet(f'{RUN_PREFIX}/train_states.parquet')
    states['timestamp'] = pd.to_datetime(states['timestamp'], utc=True)
except Exception as exc:
    states = pd.DataFrame()
    print('states not loaded:', repr(exc))

row = result.iloc[0]
features = json.loads(row['features'])
print('features:', features)
result.T


## Summary metrics

In [ ]:
summary_columns = [
    'n_components', 'covariance_type', 'random_state', 'n_rows', 'n_features',
    'n_iter', 'converged', 'train_seconds', 'log_likelihood', 'aic', 'bic', 'min_covar', 'tol'
]
result.loc[:, summary_columns]


In [ ]:
occupancy = np.asarray(json.loads(row['state_occupancy']), dtype='float64')
mean_duration = np.asarray(json.loads(row['state_mean_duration']), dtype='float64')
transition = np.asarray(json.loads(row['transition_matrix']), dtype='float64')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
states_idx = np.arange(len(occupancy))
axes[0].bar(states_idx, occupancy, color='#4C78A8')
axes[0].set_title('State occupancy')
axes[0].set_xlabel('state')
axes[0].set_ylabel('share')
axes[0].set_xticks(states_idx)

axes[1].bar(states_idx, mean_duration, color='#F58518')
axes[1].set_title('Mean state duration')
axes[1].set_xlabel('state')
axes[1].set_ylabel('minutes')
axes[1].set_xticks(states_idx)
plt.tight_layout()


## Transition matrix

In [ ]:
transition_frame = pd.DataFrame(
    transition,
    index=[f'from_{i}' for i in range(transition.shape[0])],
    columns=[f'to_{i}' for i in range(transition.shape[1])],
)
display(transition_frame.style.format('{:.4f}').background_gradient(cmap='Blues', axis=None))

fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(transition, vmin=0, vmax=1, cmap='Blues')
ax.set_title('Transition matrix')
ax.set_xlabel('to state')
ax.set_ylabel('from state')
ax.set_xticks(range(transition.shape[1]))
ax.set_yticks(range(transition.shape[0]))
for i in range(transition.shape[0]):
    for j in range(transition.shape[1]):
        ax.text(j, i, f'{transition[i, j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()


## State profiles on original features

In [ ]:
display(profiles)

mean_columns = ['state'] + [f'{feature}_mean' for feature in features]
profile_means = profiles.loc[:, mean_columns].set_index('state')
display(profile_means.style.format('{:.6g}').background_gradient(cmap='RdBu_r', axis=0))


In [ ]:
fig, axes = plt.subplots(len(features), 1, figsize=(12, 2.8 * len(features)), sharex=True)
if len(features) == 1:
    axes = [axes]
for axis, feature in zip(axes, features):
    axis.bar(profiles['state'], profiles[f'{feature}_mean'], color='#54A24B')
    axis.set_title(feature)
    axis.set_ylabel('mean')
    axis.set_xticks(profiles['state'])
axes[-1].set_xlabel('state')
plt.tight_layout()


## Duration distributions

In [ ]:
duration_stats = pd.DataFrame(json.loads(row['state_duration_stats']))
duration_stats.insert(0, 'state', range(len(duration_stats)))
display(duration_stats)

durations = json.loads(row['state_duration_distribution'])
fig, axes = plt.subplots(1, len(durations), figsize=(4 * len(durations), 3), sharey=False)
if len(durations) == 1:
    axes = [axes]
for state, axis in enumerate(axes):
    values = np.asarray(durations[state], dtype='float64')
    if len(values):
        upper = np.quantile(values, 0.99)
        axis.hist(values[values <= upper], bins=40, color='#B279A2')
    axis.set_title(f'state {state}')
    axis.set_xlabel('duration, min')
plt.tight_layout()


## State timeline

In [ ]:
if states.empty:
    print('train_states.parquet was not saved for this run')
else:
    daily = (
        states.assign(date=states['timestamp'].dt.floor('D'))
        .groupby(['date', 'state'])
        .size()
        .unstack(fill_value=0)
    )
    daily_share = daily.div(daily.sum(axis=1), axis=0)
    display(daily_share.tail())

    fig, ax = plt.subplots(figsize=(14, 5))
    daily_share.plot.area(ax=ax, colormap='tab10')
    ax.set_title('Daily state share')
    ax.set_xlabel('date')
    ax.set_ylabel('share')
    ax.legend(title='state', ncol=4, loc='upper left')
    plt.tight_layout()
